In [ ]:
import pandas as pd

player_raw = pd.read_csv("2022-2023 Football Player Stats.csv", delimiter=';', encoding='latin1')

pd.set_option('display.max_columns', None)
print(player_raw.head())


   Rk             Player Nation   Pos         Squad            Comp  Age  \
0   1   Brenden Aaronson    USA  MFFW  Leeds United  Premier League   22   
1   2   Yunis Abdelhamid    MAR    DF         Reims         Ligue 1   35   
2   3      Himad Abdelli    FRA  MFFW        Angers         Ligue 1   23   
3   4  Salis Abdul Samed    GHA    MF          Lens         Ligue 1   22   
4   5    Laurent Abergel    FRA    MF       Lorient         Ligue 1   30   

   Born  MP  Starts   Min   90s  Goals  Shots   SoT  SoT%  G/Sh  G/SoT  \
0  2000  20      19  1596  17.7      1   1.53  0.28  18.5  0.04   0.20   
1  1987  22      22  1980  22.0      0   0.86  0.05   5.3  0.00   0.00   
2  1999  14       8   770   8.6      0   1.05  0.35  33.3  0.00   0.00   
3  2000  20      20  1799  20.0      1   0.60  0.15  25.0  0.08   0.33   
4  1993  15      15  1165  12.9      0   0.31  0.00   0.0  0.00   0.00   

   ShoDist  ShoFK  ShoPK  PKatt  PasTotCmp  PasTotAtt  PasTotCmp%  PasTotDist  \
0     19.0   0.11

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Your list of player names
# player_list = ["Brenden Aaronson", "Yunis Abdelhamid", "Himad Abdelli"]
player_list = player_raw['Player'].unique()
# player_list = ['Yunis Abdelhamid']

# Base URL for player search
base_url = "https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche?query="

# DataFrame to store results
results = []

# Total number of players
total_players = len(player_list)

# Iterate through the player list with a counter
for i, player in enumerate(player_list, start=1):
    # Format the player name for the URL
    search_url = base_url + player.replace(" ", "+")
    print(f"Processing player {i}/{total_players}: {player}")

    try:
        # Fetch the page
        response = requests.get(search_url, headers={"User-Agent": "Mozilla/5.0"})
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        # Extract the market value
        market_value_tag = soup.find("td", class_="rechts hauptlink")  # Target the specific class
        if market_value_tag:
            market_value = market_value_tag.text.strip()
        else:
            market_value = "Not Found"

        # Append the result
        results.append({"Player": player, "Market Value": market_value})
    except Exception as e:
        results.append({"Player": player, "Market Value": "Error: " + str(e)})

    # Pause to avoid being blocked by the website
    time.sleep(2)

# Convert results to DataFrame
df_results = pd.DataFrame(results)

# Display or save the DataFrame
#print(df_results)
df_results.to_csv("player_market_values.csv", index=False)
print("scrapping completed")
# Website: https://www.transfermarkt.com/

Processing player 1/2530: Brenden Aaronson
Processing player 2/2530: Yunis Abdelhamid
Processing player 3/2530: Himad Abdelli
Processing player 4/2530: Salis Abdul Samed
Processing player 5/2530: Laurent Abergel
Processing player 6/2530: Oliver Abildgaard
Processing player 7/2530: Matthis Abline
Processing player 8/2530: Abner
Processing player 9/2530: Zakaria Aboukhlal
Processing player 10/2530: Tammy Abraham
Processing player 11/2530: Francesco Acerbi
Processing player 12/2530: Mohamed Achi
Processing player 13/2530: Marcos Acuña
Processing player 14/2530: Che Adams
Processing player 15/2530: Tyler Adams
Processing player 16/2530: Sargis Adamyan
Processing player 17/2530: Tosin Adarabioyo
Processing player 18/2530: Martin Adeline
Processing player 19/2530: Karim Adeyemi
Processing player 20/2530: Amine Adli
Processing player 21/2530: Yacine Adli
Processing player 22/2530: Michel Aebischer
Processing player 23/2530: Felix Afena-Gyan
Processing player 24/2530: Emmanuel Agbadou
Processi

In [ ]:
# Loading the data back in
df_results_2 = pd.read_csv("player_market_values.csv")

# Removing the data that we could not pull Market Value
df_results_2 = df_results_2[df_results_2['Market Value'] != 'Not Found']
df_results_2 = df_results_2[df_results_2['Market Value'] != '-']

# Converting the Ks and Ms to actual Numbers

# Conversion function
def convert_market_value(value):
    value = value.replace("€", "").replace(",", "").strip()  # Remove currency symbol and commas
    if "m" in value:
        return float(value.replace("m", "")) * 1_000_000  # Convert millions to numeric
    elif "k" in value:
        return float(value.replace("k", "")) * 1_000  # Convert thousands to numeric
    else:
        return float(value)  # Return the number as-is if no unit

# Apply the conversion
df_results_2["Market Value Euros"] = df_results_2["Market Value"].apply(convert_market_value)

# Joining the Market Value to the Main DF
print(player_raw.shape)
player_raw = player_raw.merge(df_results_2, how = 'left', on = 'Player')
print(player_raw.shape)

# filtering out players that we don't have market value
player_raw = player_raw[~player_raw['Market Value Euros'].isna()]
print(player_raw.shape)
player_raw.head()

(2689, 124)
(2689, 126)
(2415, 126)


,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,Min,90s,Goals,Shots,SoT,SoT%,G/Sh,G/SoT,ShoDist,ShoFK,ShoPK,PKatt,PasTotCmp,PasTotAtt,PasTotCmp%,PasTotDist,PasTotPrgDist,PasShoCmp,PasShoAtt,PasShoCmp%,PasMedCmp,PasMedAtt,PasMedCmp%,PasLonCmp,PasLonAtt,PasLonCmp%,Assists,PasAss,Pas3rd,PPA,CrsPA,PasProg,PasAtt,PasLive,PasDead,PasFK,TB,Sw,PasCrs,TI,CK,CkIn,CkOut,CkStr,PasCmp,PasOff,PasBlocks,SCA,ScaPassLive,ScaPassDead,ScaDrib,ScaSh,ScaFld,ScaDef,GCA,GcaPassLive,GcaPassDead,GcaDrib,GcaSh,GcaFld,GcaDef,Tkl,TklWon,TklDef3rd,TklMid3rd,TklAtt3rd,TklDri,TklDriAtt,TklDri%,TklDriPast,Blocks,BlkSh,BlkPass,Int,Tkl+Int,Clr,Err,Touches,TouDefPen,TouDef3rd,TouMid3rd,TouAtt3rd,TouAttPen,TouLive,ToAtt,ToSuc,ToSuc%,ToTkl,ToTkl%,Carries,CarTotDist,CarPrgDist,CarProg,Car3rd,CPA,CarMis,CarDis,Rec,RecProg,CrdY,CrdR,2CrdY,Fls,Fld,Off,Crs,TklW,PKwon,PKcon,OG,Recov,AerWon,AerLost,AerWon%,Market Value,Market Value Euros
0,1,Brenden Aaronson,USA,MFFW,Leeds United,Premier League,22,2000,20,19,1596,17.7,1,1.53,0.28,18.5,0.04,0.20,19.0,0.11,0.0,0.0,23.2,31.0,74.9,293.0,85.7,13.3,16.2,81.9,5.93,7.74,76.6,0.90,2.37,38.1,0.11,1.75,1.75,0.45,0.11,3.22,31.0,28.1,2.88,0.96,0.17,0.00,2.54,0.23,1.47,0.68,0.62,0.06,23.2,0.00,0.85,3.62,2.37,0.56,0.23,0.11,0.28,0.06,0.28,0.17,0.0,0.0,0.06,0.0,0.06,1.64,0.51,0.45,0.90,0.28,0.51,1.47,34.6,0.96,1.69,0.11,1.58,0.06,1.69,0.28,0.06,44.0,0.40,4.35,19.0,21.50,2.49,44.0,3.73,1.19,31.8,1.75,47.0,26.7,136.1,56.6,1.53,1.07,0.40,2.60,3.11,30.2,5.65,0.11,0.0,0.0,0.62,2.26,0.17,2.54,0.51,0.0,0.0,0.00,4.86,0.34,1.19,22.2,€15.00m,15000000.0
1,2,Yunis Abdelhamid,MAR,DF,Reims,Ligue 1,35,1987,22,22,1980,22.0,0,0.86,0.05,5.3,0.00,0.00,13.5,0.00,0.0,0.0,38.5,47.2,81.5,751.5,318.5,10.9,12.9,84.5,23.20,25.70,90.1,3.77,7.00,53.9,0.05,0.27,2.91,0.09,0.00,4.50,47.2,43.3,3.73,3.32,0.00,0.55,0.18,0.09,0.00,0.00,0.00,0.00,38.5,0.23,0.59,1.14,0.86,0.00,0.00,0.18,0.00,0.09,0.18,0.14,0.0,0.0,0.05,0.0,0.00,2.50,1.59,1.45,1.00,0.05,1.32,1.68,78.4,0.36,2.23,0.77,1.45,2.00,4.50,2.91,0.05,59.2,6.23,27.50,29.5,2.73,1.09,59.2,0.68,0.32,46.7,0.36,53.3,40.0,234.2,125.0,0.55,0.23,0.09,0.73,0.68,34.5,0.23,0.09,0.0,0.0,1.32,0.50,0.05,0.18,1.59,0.0,0.0,0.00,6.64,2.18,1.23,64.0,€400k,400000.0
2,3,Himad Abdelli,FRA,MFFW,Angers,Ligue 1,23,1999,14,8,770,8.6,0,1.05,0.35,33.3,0.00,0.00,19.2,0.00,0.0,0.0,40.0,49.5,80.8,676.0,188.1,18.5,22.0,84.1,15.50,18.70,82.6,4.42,5.93,74.5,0.00,1.51,3.95,1.74,0.35,6.40,49.5,48.1,1.16,0.35,0.12,0.23,1.05,0.81,0.00,0.00,0.00,0.00,40.0,0.23,1.16,2.67,2.44,0.00,0.00,0.12,0.00,0.12,0.00,0.00,0.0,0.0,0.00,0.0,0.00,2.91,1.40,1.28,1.40,0.23,1.63,2.67,60.9,1.05,1.51,0.12,1.40,0.93,3.84,0.93,0.00,62.6,0.93,11.40,36.0,17.40,1.16,62.6,3.84,2.09,54.5,1.51,39.4,48.5,298.5,151.0,2.56,2.56,0.47,2.09,1.05,43.4,5.93,0.12,0.0,0.0,1.74,1.28,0.00,1.05,1.40,0.0,0.0,0.00,8.14,0.93,1.05,47.1,€7.00m,7000000.0
3,4,Salis Abdul Samed,GHA,MF,Lens,Ligue 1,22,2000,20,20,1799,20.0,1,0.60,0.15,25.0,0.08,0.33,20.3,0.00,0.0,0.0,59.5,64.9,91.6,946.3,226.9,29.6,31.8,93.2,24.70,26.20,94.3,3.35,4.30,77.9,0.00,0.50,6.00,0.55,0.10,5.60,64.9,63.1,1.40,1.30,0.05,0.20,0.35,0.10,0.00,0.00,0.00,0.00,59.5,0.35,0.40,1.60,1.35,0.00,0.10,0.10,0.00,0.05,0.00,0.00,0.0,0.0,0.00,0.0,0.00,1.50,0.80,0.55,0.85,0.10,0.85,1.30,65.4,0.45,1.30,0.35,0.95,1.10,2.60,0.80,0.00,73.3,2.10,12.00,49.6,12.20,0.70,73.3,1.25,0.70,56.0,0.40,32.0,61.0,316.9,117.5,1.25,1.95,0.15,1.35,1.30,56.5,1.70,0.15,0.0,0.0,2.45,1.35,0.00,0.35,0.80,0.0,0.0,0.05,6.60,0.50,0.50,50.0,€5.00m,5000000.0
4,5,Laurent Abergel,FRA,MF,Lorient,Ligue 1,30,1993,15,15,1165,12.9,0,0.31,0.00,0.0,0.00,0.00,23.9,0.00,0.0,0.0,37.9,43.4,87.3,613.6,224.7,17.9,19.4,92.4,15.70,17.80,88.6,2.64,3.95,66.7,0.08,0.62,3.88,0.39,0.00,5.04,43.4,42.6,0.78,0.78,0.39,0.16,0.23,0.00,0.00,0.00,0.00,0.00,37.9,0.08,0.31,1.24,1.01,0.08,0.00,0.08,0.00,0.08,0.08,0.08,0.0,0.0,0.00,0.0,0.00,3.80,2.02,2.64,1.16,0.00,1.32,3.26,40.5,1.94,1.40,0.23,1.16,1.16,4.96,1.55,0.00,54.7,3.26,19.20,31.4,4.88,0.23,54.7,0.93,0.54,58.3,0.31,33.3,41.0,174.3,72.7,0.47,0.93,0.

In [ ]:
import pandas as pd

# Load the scraped market value data
df_results_2 = pd.read_csv("player_market_values.csv")

# Preview columns
print("Columns in scraped data:", df_results_2.columns.tolist())

# Clean invalid entries
df_results_2 = df_results_2[
    (df_results_2['Market Value'].notna()) &
    (~df_results_2['Market Value'].isin(['Not Found', '-']))
]

# Conversion function (robust)
def convert_market_value(value):
    try:
        value = str(value).replace("€", "").replace(",", "").strip().lower()
        if "m" in value:
            return float(value.replace("m", "")) * 1_000_000
        elif "k" in value:
            return float(value.replace("k", "")) * 1_000
        else:
            return float(value)
    except:
        return None  # fallback if conversion fails

# Apply conversion and verify
df_results_2["Market Value Euros"] = df_results_2["Market Value"].apply(convert_market_value)

print("✅ Conversion done. Non-null count:", df_results_2["Market Value Euros"].notna().sum())

# Merge with player_raw
print("Before merge:", player_raw.shape)
player_raw = player_raw.merge(df_results_2, how="left", on="Player")
print("After merge:", player_raw.shape)

# Check if the column exists after merge
if "Market Value Euros" in player_raw.columns:
    player_raw = player_raw[~player_raw["Market Value Euros"].isna()]
    print("After filtering:", player_raw.shape)
else:
    print("⚠️ 'Market Value Euros' column missing after merge!")

# Save cleaned dataset
player_raw.to_csv("player_data_with_market_value.csv", index=False)
print("✅ Saved: player_data_with_market_value.csv")


Columns in scraped data: ['Player', 'Market Value']
✅ Conversion done. Non-null count: 2266
Before merge: (2415, 128)
After merge: (2415, 130)
After filtering: (2415, 130)
✅ Saved: player_data_with_market_value.csv
